In [62]:
from google.colab import files
import pyspark

In [64]:
from pyspark.sql import SparkSession

In [65]:
spark = SparkSession.builder\
    .appName("Uber_analysis")\
        .getOrCreate()

In [66]:
df =spark.read.csv('dataset.csv',header = True,inferSchema=True)

In [67]:
df.show()

+---------+------------+---------+-------+----------------+---------+--------------+
|     Date|Time (Local)|Eyeballs |Zeroes |Completed Trips |Requests |Unique Drivers|
+---------+------------+---------+-------+----------------+---------+--------------+
|10-Sep-12|           7|        5|      0|               2|        2|             9|
|     NULL|           8|        6|      0|               2|        2|            14|
|     NULL|           9|        8|      3|               0|        0|            14|
|     NULL|          10|        9|      2|               0|        1|            14|
|     NULL|          11|       11|      1|               4|        4|            11|
|     NULL|          12|       12|      0|               2|        2|            11|
|     NULL|          13|        9|      1|               0|        0|             9|
|     NULL|          14|       12|      1|               0|        0|             9|
|     NULL|          15|       11|      2|               1|      

In [68]:
from pyspark.sql import *
from pyspark.sql.functions import *

In [69]:
df_updated =df.withColumn('rowid',monotonically_increasing_id())

In [70]:
df_updated.show()

+---------+------------+---------+-------+----------------+---------+--------------+-----+
|     Date|Time (Local)|Eyeballs |Zeroes |Completed Trips |Requests |Unique Drivers|rowid|
+---------+------------+---------+-------+----------------+---------+--------------+-----+
|10-Sep-12|           7|        5|      0|               2|        2|             9|    0|
|     NULL|           8|        6|      0|               2|        2|            14|    1|
|     NULL|           9|        8|      3|               0|        0|            14|    2|
|     NULL|          10|        9|      2|               0|        1|            14|    3|
|     NULL|          11|       11|      1|               4|        4|            11|    4|
|     NULL|          12|       12|      0|               2|        2|            11|    5|
|     NULL|          13|        9|      1|               0|        0|             9|    6|
|     NULL|          14|       12|      1|               0|        0|             9|    7|

In [71]:
window_spec = Window.orderBy('rowid').rowsBetween(Window.unboundedPreceding,0)

In [72]:
# Forward fill the Date column
df_updated = df_updated.withColumn(
    "Date",
    last(col("Date"), ignorenulls=True).over(window_spec)
)


In [73]:
df_updated = df_updated.drop('rowid')

Trend of request in the dataset?


In [74]:
from pyspark.sql import functions as F

In [75]:
df_total_req = df_updated.groupBy('Date').agg(F.sum('Requests '))

In [76]:
daily_req=df_total_req

In [77]:
min_date = daily_req.agg(min("Date")).first()[0]

daily_requests = daily_req.withColumn(
    "day_index",
    datediff(to_date(col("Date"), "dd-MMM-yy"), to_date(lit(min_date), "dd-MMM-yy")) + 1
)

In [78]:
daily_requests.show()

+---------+--------------+---------+
|     Date|sum(Requests )|day_index|
+---------+--------------+---------+
|10-Sep-12|            34|        1|
|11-Sep-12|            52|        2|
|12-Sep-12|           114|        3|
|13-Sep-12|            67|        4|
|14-Sep-12|           137|        5|
|15-Sep-12|           282|        6|
|16-Sep-12|           118|        7|
|17-Sep-12|            78|        8|
|18-Sep-12|            81|        9|
|19-Sep-12|            54|       10|
|20-Sep-12|            95|       11|
|21-Sep-12|           240|       12|
|22-Sep-12|           344|       13|
|23-Sep-12|           154|       14|
|24-Sep-12|             8|       15|
+---------+--------------+---------+



In [79]:
correlation = daily_requests.select(
    corr("day_index", "sum(Requests )").alias("trend_correlation")
).first()["trend_correlation"]

In [80]:
if correlation > 0:
    print(f" INCREASING trend (correlation: {correlation:.2f})")
elif correlation < 0:
    print(f" DECREASING trend (correlation: {correlation:.2f})")
else:
    print(f" NO trend (correlation: {correlation:.2f})")

 INCREASING trend (correlation: 0.28)


##

Which time slots (morning/afternoon/evening/night) are busiest?

root
 |-- Date: string (nullable = true)
 |-- Time (Local): integer (nullable = true)
 |-- Eyeballs : integer (nullable = true)
 |-- Zeroes : integer (nullable = true)
 |-- Completed Trips : integer (nullable = true)
 |-- Requests : integer (nullable = true)
 |-- Unique Drivers: integer (nullable = true)



In [82]:
df_with_date = df_updated.withColumn(
    "Time (Local)",
    col("Time (Local)").cast("int")
)

In [83]:
df_time_slot = df_with_date.withColumn(
    "Slot",
    when(col("Time (Local)").between(0, 6), "Night")
    .when(col("Time (Local)").between(6, 12), "Morning")
    .when(col("Time (Local)").between(12, 17), "Afternoon")
    .when(col("Time (Local)").between(17, 24), "Evening")
    .otherwise("Night")
)


In [88]:
df_time = df_time_slot.groupBy('Slot').agg(F.sum('Requests '))

In [89]:
df_time.orderBy('sum(Requests )',ascending=False).select('Slot').show()

+---------+
|     Slot|
+---------+
|  Evening|
|    Night|
|Afternoon|
|  Morning|
+---------+





Compare weekday vs weekend performance

In [92]:
df_updated = df_updated.withColumn(
    "Date",
    to_date(col("Date"), "dd-MMM-yy")
)

In [99]:
df_week = df_updated.withColumn('day_of_week',when((dayofweek(col("Date")) ==1) | (dayofweek(col("Date")) ==7) ,"Weekend").otherwise("Weekday"))

In [100]:
df_week.show()

+----------+------------+---------+-------+----------------+---------+--------------+-----------+
|      Date|Time (Local)|Eyeballs |Zeroes |Completed Trips |Requests |Unique Drivers|day_of_week|
+----------+------------+---------+-------+----------------+---------+--------------+-----------+
|2012-09-10|           7|        5|      0|               2|        2|             9|    Weekday|
|2012-09-10|           8|        6|      0|               2|        2|            14|    Weekday|
|2012-09-10|           9|        8|      3|               0|        0|            14|    Weekday|
|2012-09-10|          10|        9|      2|               0|        1|            14|    Weekday|
|2012-09-10|          11|       11|      1|               4|        4|            11|    Weekday|
|2012-09-10|          12|       12|      0|               2|        2|            11|    Weekday|
|2012-09-10|          13|        9|      1|               0|        0|             9|    Weekday|
|2012-09-10|        

In [102]:
df_week.groupBy('day_of_week').agg(F.sum('Completed Trips ')).show()

+-----------+---------------------+
|day_of_week|sum(Completed Trips )|
+-----------+---------------------+
|    Weekday|                  714|
|    Weekend|                  651|
+-----------+---------------------+





Which day of the week (Mon-Sun) had the highest average completed trips?

In [103]:
df_updated.printSchema()

root
 |-- Date: date (nullable = true)
 |-- Time (Local): integer (nullable = true)
 |-- Eyeballs : integer (nullable = true)
 |-- Zeroes : integer (nullable = true)
 |-- Completed Trips : integer (nullable = true)
 |-- Requests : integer (nullable = true)
 |-- Unique Drivers: integer (nullable = true)



In [128]:
df_day_of_week = df_updated.withColumn('day_of_week',date_format(col('Date'),'EEEE'))

In [129]:
df_day_of_week.show()

+----------+------------+---------+-------+----------------+---------+--------------+-----------+
|      Date|Time (Local)|Eyeballs |Zeroes |Completed Trips |Requests |Unique Drivers|day_of_week|
+----------+------------+---------+-------+----------------+---------+--------------+-----------+
|2012-09-10|           7|        5|      0|               2|        2|             9|     Monday|
|2012-09-10|           8|        6|      0|               2|        2|            14|     Monday|
|2012-09-10|           9|        8|      3|               0|        0|            14|     Monday|
|2012-09-10|          10|        9|      2|               0|        1|            14|     Monday|
|2012-09-10|          11|       11|      1|               4|        4|            11|     Monday|
|2012-09-10|          12|       12|      0|               2|        2|            11|     Monday|
|2012-09-10|          13|        9|      1|               0|        0|             9|     Monday|
|2012-09-10|        

In [130]:
df_day_of_week= df_day_of_week.groupBy('day_of_week').agg(F.avg('Completed Trips '))

In [132]:
answer =  df_day_of_week.orderBy('avg(Completed Trips )',ascending=False).select('day_of_week').first()['day_of_week']

In [133]:
print(answer)

Saturday
